# Environment Variable Utilities API Reference

Developer-facing statements defined in `libs/core/langchain_core/utils/env.py`.

# `env_var_is_set`

Checks whether an environment variable exists and has an enabled value.

```python
env_var_is_set(
    env_var: str, # Name of the environment variable
) -> bool # Whether the variable is set to an enabled value
```

Returns `False` when the variable is absent or its exact value is `""`, `"0"`, `"false"`, or `"False"`. Other values return `True`.

---

# `get_from_dict_or_env`

Gets a value from a dictionary, then falls back to an environment variable and optional default.

```python
get_from_dict_or_env(
    data: dict[str, Any], # Dictionary to search
    key: str | list[str], # Dictionary key or ordered keys to try
    env_key: str, # Environment variable used when no dictionary value is found
    default: str | None = None, # Value used when neither source provides a value
) -> str # Selected value converted to a string
```

When `key` is a list, keys are checked in order and the first truthy dictionary value is returned. A single key is also used only when its dictionary value is truthy. Dictionary values are converted with `str`.

When no dictionary value is found, the function delegates to `get_from_env`. For a list of keys, the first key is used in the fallback error message.

Raises `ValueError` through `get_from_env` when no truthy environment value exists and `default` is `None`.

---

# `get_from_env`

Gets a value from an environment variable or returns a default.

```python
get_from_env(
    key: str, # Parameter name used in the error message
    env_key: str, # Environment variable to read
    default: str | None = None, # Value returned when the environment variable is unavailable
) -> str # Environment value or default
```

Returns the environment value when `os.getenv(env_key)` is truthy. Otherwise, it returns `default` when the default is not `None`.

Raises `ValueError` when the environment variable is absent or empty and no default is provided.

In [ ]:
import os # Import operating-system utilities for environment variables

from langchain_core.utils.env import env_var_is_set # Import the environment-variable status checker
from langchain_core.utils.env import get_from_dict_or_env # Import dictionary and environment lookup utility
from langchain_core.utils.env import get_from_env # Import environment-variable lookup utility

In [ ]:
# 1. Check whether an environment variable is enabled
os.environ["LANGCHAIN_DEBUG"] = "false" # Set the variable to a disabled value

is_enabled = env_var_is_set("LANGCHAIN_DEBUG") # Check the variable again

print(is_enabled) # Display False

# Values such as "", "0", "false", and "False" are treated as disabled.
os.environ["LANGCHAIN_DEBUG"] = "false" # Set the variable to a disabled value

is_enabled = env_var_is_set("LANGCHAIN_DEBUG") # Check the variable again

print(is_enabled) # Display False

In [ ]:
# 2. Read a value from an environment variable
os.environ["OPENAI_API_KEY"] = "demo-api-key" # Create an example API-key environment variable

api_key = get_from_env( # Retrieve the API key
    key="api_key", # Provide the parameter name used in error messages
    env_key="OPENAI_API_KEY", # Specify the environment variable to read
) # Finish retrieving the value

print(api_key) # Display the environment-variable value

# A default can be supplied when the variable does not exist:
model_name = get_from_env( # Retrieve a model name
    key="model_name", # Provide the parameter name used in error messages
    env_key="MODEL_NAME", # Try to read this environment variable
    default="gpt-4.1-mini", # Use this value when the variable is unavailable
) # Finish retrieving the value

print(model_name) # Display the default model name


In [ ]:
# 3. Prefer dictionary values over environment variables
os.environ["MODEL_NAME"] = "model-from-environment" # Define the fallback environment value

config = { # Create an application configuration dictionary
    "model": "model-from-dictionary", # Provide the preferred dictionary value
} # Finish creating the dictionary

model_name = get_from_dict_or_env( # Retrieve the model configuration
    data=config, # Search this dictionary first
    key="model", # Look for this dictionary key
    env_key="MODEL_NAME", # Fall back to this environment variable
) # Finish retrieving the value

print(model_name) # Display model-from-dictionary

In [ ]:
# 4. Try multiple dictionary keys in order
config = { # Create a configuration containing an alternative key
    "model_name": "", # Provide an empty value that will be ignored
    "model": "gpt-4.1", # Provide the first truthy matching value
} # Finish creating the dictionary

model_name = get_from_dict_or_env( # Retrieve a value using multiple possible keys
    data=config, # Search this configuration dictionary
    key=["model_name", "model"], # Check these keys from left to right
    env_key="MODEL_NAME", # Use the environment variable when neither key has a truthy value
) # Finish retrieving the value

print(model_name) # Display gpt-4.1

In [ ]:
# 5. Handle a missing required value
os.environ.pop("REQUIRED_TOKEN", None) # Ensure the environment variable does not exist

try: # Begin exception handling
    token = get_from_env( # Try to retrieve a required value
        key="token", # Provide the parameter name
        env_key="REQUIRED_TOKEN", # Specify the missing environment variable
    ) # Finish the lookup

except ValueError as error: # Catch the missing-value error
    print(error) # Display the generated error message